In [ ]:
import pandas as pd

sheets = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None, dtype={"Invoice": str, "StockCode": str, "Description": str ,"Customer ID": str})

print(type(sheets))          # <class 'dict'>
print(sheets.keys())         # dict_keys(['2023', '2024'])

for name, frame in sheets.items():
    print(name, frame.shape)

<class 'dict'>
dict_keys(['Year 2009-2010', 'Year 2010-2011'])
Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)


In [46]:
from pathlib import Path
import numpy as np

# --- Paths: find the project root, so it works from notebooks/ or the root ---
def project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / ".git").exists():
            return p
    raise FileNotFoundError("No .git folder found above " + str(start))

print(project_root)

<function project_root at 0x116370f60>


In [47]:
ROOT = project_root()
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
PROCESSED.mkdir(parents=True, exist_ok=True)   # create if missing, no error if present
REPORTS.mkdir(parents=True, exist_ok=True)

In [40]:
# Stack vertically and reset index
combined_df = pd.concat([sheets['Year 2009-2010'], sheets['Year 2010-2011']], ignore_index=True)
print(combined_df.head)
print(combined_df.tail)

<bound method NDFrame.head of         Invoice StockCode                          Description  Quantity  \
0        489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1        489434    79323P                   PINK CHERRY LIGHTS        12   
2        489434    79323W                  WHITE CHERRY LIGHTS        12   
3        489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4        489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...         ...       ...                                  ...       ...   
1067366  581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
1067368  581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
1067369  581587     22138        BAKING SET 9 PIECE RETROSPOT          3   
1067370  581587      POST                              POSTAGE         1   

                InvoiceDate  Price Customer ID         Co

In [13]:
print(sheets['Year 2009-2010'].columns.equals((sheets['Year 2010-2011'].columns)))

True


In [16]:
print(sheets['Year 2009-2010'].head(5))
print(sheets['Year 2010-2011'].head(5))

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
  Invoice StockCode                          Description  Quantity  \
0  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1  536365     71053                  WHITE METAL LANTERN         6   
2  536365

In [ ]:
first_sheet_daterange = sheets["Year 2009-2010"]["InvoiceDate"].max() - sheets['Year 2009-2010']['InvoiceDate'].min()
print(first_sheet_daterange)
second_sheet_daterange = sheets["Year 2010-2011"]["InvoiceDate"].max() - sheets['Year 2010-2011']['InvoiceDate'].min()
print(second_sheet_daterange)
#sheets['Year 2009-2010'].columns.tolist()
# sheets['Year 2009-2010']['Description'] 
# print(sheets["Year 2009-2010"].dtypes)
# print(sheets["Year 2010-2011"].dtypes)
print(combined_df.dtypes)

373 days 12:16:00
373 days 04:24:00
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object
Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object


In [45]:
# toy.dtypes                 # dtype
# toy.notna().sum()          # non_null  (toy.count() does the same)
# toy.isna().sum()           # missing
# toy.isna().mean() * 100    # missing_%  (mean of True/False = fraction True)
# toy.nunique()              # n_unique, excludes NaN by default

def df_summary(df):
    print("datatypes: ----------------\n", df.dtypes)
    print("row count:---------------- \n", df.shape[0])
    print("Not NA: -----------------\n", df.notna().sum())
    print("NA count ----------------\n", df.isna().sum())
    print("missing_%: --------------\n", df.isna().mean() * 100)
    print("Unique columns ----------------\n", df.nunique())

#df_summary(sheets["Year 2009-2010"])
# df_summary(sheets["Year 2010-2011"])
df_summary(combined_df)




datatypes: ----------------
 Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object
row count:---------------- 
 1067371
Not NA: -----------------
 Invoice        1067371
StockCode      1067371
Description    1062989
Quantity       1067371
InvoiceDate    1067371
Price          1067371
Customer ID     824364
Country        1067371
dtype: int64
NA count ----------------
 Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
missing_%: --------------
 Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
dtype: float6

In [ ]:
def apply_rule(df, keep_fn, rule, log):
    """Keep rows where keep_fn(df) is True; record what was removed."""
    before = len(df)
    out = df[keep_fn(df)].copy()
    log.append({
        "rule": rule,
        "rows_before": before,
        "removed": before - len(out),
        "pct_removed": round(100 * (before - len(out)) / before, 2),
        "rows_after": len(out),
    })
    return out    

def clean_data(raw):
    log = []
    df = raw.copy()
    df["StockCode"] = df["StockCode"].str.strip().str.upper()    # normalise, no removal

    df = apply_rule(df, lambda d: d["Customer ID"].notna(), "Customer ID", log)
    df = apply_rule(df, lambda d: ~d["Invoice"].str.startswith("C"), "Cancellations", log)
    df = apply_rule(df, lambda d: d["Quantity"] > 0, "Quantity <= 0", log)
    df = apply_rule(df, lambda d: d["Price"] > 0, "Price <= 0", log)  
    # df = apply_rule(df, lambda d: ~d["Invoice"].str.startswith("C"), "Cancellations", log)
    return df, pd.DataFrame(log)

clean_df, log_df = clean_data(combined_df)
print(log_df)

            rule  rows_before  removed  pct_removed  rows_after
0    Customer ID      1067371   243007        22.77      824364
1  Quantity <= 0       824364    18744         2.27      805620
2     Price <= 0       805620       71         0.01      805549
3  Cancellations       805549        0         0.00      805549


In [56]:
filtered_df = combined_df[combined_df['Invoice'].str.startswith('C')]

print(len(filtered_df))

19494
